In [83]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
# treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    use_sud=False,
    matrix_type="coverage",
    excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
        # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own"]
)

Find the already annotated polycategorial lexical units (such as divers det,adj or from adp, sconj)

In [94]:
previous_lemma_pos = list(corpus._idx2lexunit.values())[0]
polycategorial_lexunits = {}
for lemma, pos in list(corpus._idx2lexunit.values())[1:]:
    current_lemma_pos = (lemma, pos)
    if current_lemma_pos[0] == previous_lemma_pos[0]:
        if current_lemma_pos[0] not in polycategorial_lexunits:
            polycategorial_lexunits[current_lemma_pos[0]] = (current_lemma_pos[1], previous_lemma_pos[1])
        else:
            polycategorial_lexunits[current_lemma_pos[0]] += (current_lemma_pos[1],)
    previous_lemma_pos = current_lemma_pos
print(polycategorial_lexunits)
print(len(polycategorial_lexunits))
    
    

{'/': ('CCONJ', 'ADP'), '1er': ('NUM', 'ADJ'), 'allemand': ('NOUN', 'ADJ'), 'américain': ('NOUN', 'ADJ'), 'anglais': ('NOUN', 'ADJ'), 'après': ('ADV', 'ADP'), 'arabe': ('NOUN', 'ADJ'), 'aucun': ('PRON', 'DET'), 'autant': ('PRON', 'ADV'), 'autre': ('PRON', 'ADJ'), 'avant': ('ADV', 'ADP', 'NOUN'), 'avoir': ('VERB', 'AUX'), 'beaucoup': ('PRON', 'ADV'), 'bien': ('NOUN', 'ADV'), 'blanc': ('NOUN', 'ADJ'), 'bref': ('INTJ', 'ADJ'), 'britannique': ('NOUN', 'ADJ'), 'caractéristique': ('NOUN', 'ADJ'), 'ce': ('PRON', 'DET'), 'certain': ('DET', 'ADJ', 'PRON'), 'chrétien': ('NOUN', 'ADJ'), 'comme': ('CCONJ', 'ADP', 'SCONJ'), 'conseiller': ('VERB', 'NOUN'), 'courant': ('NOUN', 'ADJ'), 'critique': ('NOUN', 'ADJ'), 'depuis': ('ADV', 'ADP'), 'différent': ('DET', 'ADJ'), 'divers': ('DET', 'ADJ'), 'donc': ('CCONJ', 'ADV'), 'droit': ('NOUN', 'ADJ'), 'en': ('PRON', 'ADP'), 'ensemble': ('NOUN', 'ADV'), 'espagnol': ('NOUN', 'ADJ'), 'essentiel': ('NOUN', 'ADJ'), 'extérieur': ('NOUN', 'ADJ'), 'faire': ('VERB', 

In [95]:
pos_nbelements = {}
for lemma, pos in list(corpus._idx2lexunit.values())[0:]:
    if pos not in pos_nbelements:
        pos_nbelements[pos] = {"total": 0, "polycategorial": 0}
    pos_nbelements[pos]["total"] += 1
    if lemma in polycategorial_lexunits:
        pos_nbelements[pos]["polycategorial"] += 1
print(pos_nbelements)

{'NOUN': {'total': 1400, 'polycategorial': 59}, 'CCONJ': {'total': 14, 'polycategorial': 5}, 'ADP': {'total': 35, 'polycategorial': 7}, 'NUM': {'total': 166, 'polycategorial': 3}, 'ADJ': {'total': 402, 'polycategorial': 59}, 'VERB': {'total': 556, 'polycategorial': 6}, 'PROPN': {'total': 172, 'polycategorial': 0}, 'ADV': {'total': 123, 'polycategorial': 19}, 'DET': {'total': 15, 'polycategorial': 11}, 'PRON': {'total': 40, 'polycategorial': 16}, 'AUX': {'total': 3, 'polycategorial': 3}, 'INTJ': {'total': 2, 'polycategorial': 1}, 'SCONJ': {'total': 6, 'polycategorial': 3}, 'X': {'total': 1, 'polycategorial': 0}}


In [61]:
corpus.lexunit2idx(('soit', 'ADV'))

2506

Measure the distance between the vectors representing the two lexical units (e.g. v1 = anglais, noun and v2 = anglais, adj) and compare the cross-POS distance to the average distance within the category. A split is justified if the distance between (anglais, N) and (anglais, adj) is significantly higher than the typical distance between two nouns.

In [86]:
def _get_category_stats(target_pos, target_idx, target_vec, metric='cosine'):
        # Indices of all other members of this POS
        cat_indices = [idx for idx, unit in idx_to_unit.items() 
                       if unit[1] == target_pos and idx != target_idx]
        
        if len(cat_indices) < 2:
            return 0.0, 0.0, 0.0 # Not enough data to compute stats
        
        # Distances from our target word to all other words in its POS
        cat_matrix = matrix[cat_indices]
        dists = cdist(target_vec, cat_matrix, metric=metric)[0]
        
        return np.mean(dists), np.std(dists), len(cat_indices)

In [87]:
import numpy as np
from scipy.spatial.distance import cdist

def check_cross_pos_distance(matrix, idx_to_unit, unit_to_idx, lemma, poses, metric='cosine', k=1.0):
    """
    Checks if dist(lemma_pos1, lemma_pos2) is greater than the intra-category 
    distance for BOTH categories.
    
    Args:
        matrix (np.array): N x M matrix (rows=units, cols=features).
        idx_to_unit (dict): Maps row index -> (lemma, pos).
        unit_to_idx (dict): Maps (lemma, pos) -> row index.
        lemma (str): The lemma to check.
        poses (tuple): A tuple of two POS tags (pos1, pos2).
        metric (str): Distance metric ('cosine', 'euclidean', etc.).
        
    Returns:
        dict: Results containing distances, the comparison boolean, and the farthest element.
    """
    
    target_1 = (lemma, poses[0])
    target_2 = (lemma, poses[1])

    idx1 = unit_to_idx[target_1]
    idx2 = unit_to_idx[target_2]

    vec1 = matrix[idx1].reshape(1, -1) # Reshape for cdist
    vec2 = matrix[idx2].reshape(1, -1)

    # 3. Calculate Cross-POS distance (e.g., dist(dance_N, dance_V))
    # cdist returns a 2D array, we take [0][0]
    cross_dist = cdist(vec1, vec2, metric=metric)[0][0]

    mu1, std1, count1 = _get_category_stats(poses[0], idx1, vec1, metric)
    mu2, std2, count2 = _get_category_stats(poses[1], idx2, vec2, metric)

    # 4. Calculate Z-scores
    # How many standard deviations is the cross_dist away from the category mean?
    z1 = (cross_dist - mu1) / std1 if std1 > 0 else 0
    z2 = (cross_dist - mu2) / std2 if std2 > 0 else 0

    # 5. Final Logic
    # We split if the cross_dist is significantly larger than the average variation in BOTH categories.
    # should_split = (z1 > k) or (z2 > k)
    should_split = cross_dist > mu1 or cross_dist > mu2

    return {
        "lemma": lemma,
        "pos1": poses[0],
        "pos2": poses[1],
        "cross_dist": round(float(cross_dist), 4),
        f"{poses[0]}_stats": {"mean": round(float(mu1), 4), "std": round(float(std1), 4), "z_score": round(float(z1), 2)},
        f"{poses[1]}_stats": {"mean": round(float(mu2), 4), "std": round(float(std2), 4), "z_score": round(float(z2), 2)},
        "should_split": bool(should_split),
        "threshold_k": k
    }

# --- Example Usage ---

# # 1. Create Dummy Data
# # 5 lexical units, 3 features each
matrix = np.array([
    [1.0, 0.9, 0.1],  # 0: (dance, noun)
    [0.1, 0.2, 0.9],  # 1: (dance, verb) - Very different from noun
    [0.9, 0.8, 0.2],  # 2: (table, noun) - Similar to dance_noun
    [0.8, 0.9, 0.1],  # 3: (chair, noun) - Similar to dance_noun
    [0.5, 0.5, 0.5],  # 4: (weird, noun) - The "farthest" noun
])

idx_to_unit = {
    0: ("dance", "noun"),
    1: ("dance", "verb"),
    2: ("table", "noun"),
    3: ("chair", "noun"),
    4: ("weird", "noun")
}

unit_to_idx = {v: k for k, v in idx_to_unit.items()}
# 2. Run the check
result = check_cross_pos_distance(matrix, idx_to_unit, unit_to_idx, "dance", ("noun","verb"), metric='cosine', k=1.0)

# 3. Print Results
print(result)
# print(f"Distance (dance, N) <-> (dance, V): {result['cross_dist']:.4f}")
# print(f"Max Dist (dance, N) <-> Any Noun:   {result['max_intra_cat_dist']:.4f}")
# print(f"Farthest Noun:                      {result['farthest_same_cat_unit']}")
# print("-" * 30)
# print(f"Is Cross Distance Bigger?           {result['is_cross_bigger']}")

{'lemma': 'dance', 'pos1': 'noun', 'pos2': 'verb', 'cross_dist': 0.7043, 'noun_stats': {'mean': 0.0515, 'std': 0.0655, 'z_score': 9.97}, 'verb_stats': {'mean': 0.0, 'std': 0.0, 'z_score': 0.0}, 'should_split': True, 'threshold_k': 1.0}


In [70]:
True or False

True

In [88]:
matrix = corpus.feature_matrix
idx_to_unit = corpus._idx2lexunit
unit_to_idx = corpus._lexunit2idx
for lemma, poses in polycategorial_lexunits.items():
    comparison_result = check_cross_pos_distance(matrix, idx_to_unit, unit_to_idx, lemma, poses, metric='cosine', k=1.0)
    # print(f"{comparison_result['lemma']}, {comparison_result['pos1']} vs {comparison_result['pos2']}: should split? {comparison_result['should_split']}")
    print(comparison_result)

    # print(f"{dico['lemma']}, {dico['pos1']} vs {dico['pos2']}: {decision}")

{'lemma': '/', 'pos1': 'CCONJ', 'pos2': 'ADP', 'cross_dist': 0.148, 'CCONJ_stats': {'mean': 0.3161, 'std': 0.1018, 'z_score': -1.65}, 'ADP_stats': {'mean': 0.2917, 'std': 0.0893, 'z_score': -1.61}, 'should_split': False, 'threshold_k': 1.0}
{'lemma': '1er', 'pos1': 'NUM', 'pos2': 'ADJ', 'cross_dist': 0.3058, 'NUM_stats': {'mean': 0.5565, 'std': 0.2299, 'z_score': -1.09}, 'ADJ_stats': {'mean': 0.4708, 'std': 0.1825, 'z_score': -0.9}, 'should_split': False, 'threshold_k': 1.0}
{'lemma': 'allemand', 'pos1': 'NOUN', 'pos2': 'ADJ', 'cross_dist': 0.5409, 'NOUN_stats': {'mean': 0.1516, 'std': 0.0748, 'z_score': 5.2}, 'ADJ_stats': {'mean': 0.1874, 'std': 0.2027, 'z_score': 1.74}, 'should_split': True, 'threshold_k': 1.0}
{'lemma': 'américain', 'pos1': 'NOUN', 'pos2': 'ADJ', 'cross_dist': 0.7016, 'NOUN_stats': {'mean': 0.2184, 'std': 0.0983, 'z_score': 4.92}, 'ADJ_stats': {'mean': 0.1931, 'std': 0.1979, 'z_score': 2.57}, 'should_split': True, 'threshold_k': 1.0}
{'lemma': 'anglais', 'pos1': 'NO

In [89]:
import numpy as np

def get_discriminating_features(matrix, unit_to_idx, idx_to_feature, lemma, poses, top_n=10):
    """
    Identifies which features (columns) have the largest difference between two senses.
    """
    pos1, pos2 = poses
    target_1, target_2 = (lemma, pos1), (lemma, pos2)

    # 1. Extract the vectors
    idx1 = unit_to_idx[target_1]
    idx2 = unit_to_idx[target_2]
    
    vec1 = matrix[idx1]
    vec2 = matrix[idx2]

    # 2. Calculate absolute difference per feature: |v1 - v2|
    # Formula: $\Delta f = |v_{1,f} - v_{2,f}|$
    diffs = np.abs(vec1 - vec2)

    # 3. Get indices of the largest differences
    # argsort sorts ascending, so we take the end of the array and reverse it
    top_indices = np.argsort(diffs)[-top_n:][::-1]

    results = []
    for i in top_indices:
        feature_name = idx_to_feature[i]
        results.append({
            "feature": feature_name,
            f"val_{pos1}": round(float(vec1[i]), 4),
            f"val_{pos2}": round(float(vec2[i]), 4),
            "abs_diff": round(float(diffs[i]), 4)
        })

    return results

In [90]:
top_diffs = get_discriminating_features(matrix, unit_to_idx, corpus._idx2feature, "of", ("SCONJ", "ADP"))

for d in top_diffs:
    print(d)

KeyError: ('of', 'SCONJ')

In [99]:
from itertools import combinations
import numpy as np
from scipy.spatial.distance import cdist

def check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20, metric='cosine'):
    # Generate all unique pairs of poses
    # e.g., if poses=['NOUN', 'VERB', 'ADJ'], pairs are (N, V), (N, A), (V, A)
    pose_pairs = list(combinations(poses, 2))
    results = []

    def get_local_density(target_pos, target_idx, target_vec):
        all_cat_indices = [idx for idx, u in idx_to_unit.items() if u[1] == target_pos and idx != target_idx]
        
        if not all_cat_indices:
            return 0.0
            
        if len(all_cat_indices) < k_neighbors:
            compare_indices = all_cat_indices
        else:
            all_dists = cdist(target_vec, matrix[all_cat_indices], metric=metric)[0]
            closest_k_relative_indices = np.argsort(all_dists)[:k_neighbors]
            compare_indices = [all_cat_indices[i] for i in closest_k_relative_indices]
        
        local_dists = cdist(target_vec, matrix[compare_indices], metric=metric)[0]
        return np.median(local_dists)

    # Iterate through each pair and calculate the split strength
    for pos1, pos2 in pose_pairs:
        t1, t2 = (lemma, pos1), (lemma, pos2)
        
        # Skip this pair if one of the senses isn't in our mapping
        if t1 not in unit_to_idx or t2 not in unit_to_idx:
            continue

        v1 = matrix[unit_to_idx[t1]].reshape(1, -1)
        v2 = matrix[unit_to_idx[t2]].reshape(1, -1)
        cross_dist = cdist(v1, v2, metric=metric)[0][0]

        local_size1 = get_local_density(pos1, unit_to_idx[t1], v1)
        local_size2 = get_local_density(pos2, unit_to_idx[t2], v2)

        robust_ratio1 = cross_dist / local_size1 if local_size1 > 0 else 0
        robust_ratio2 = cross_dist / local_size2 if local_size2 > 0 else 0
        split_strength = min(robust_ratio1, robust_ratio2)

        results.append({
            "lemma": lemma,
            "poses_compared": (pos1, pos2), # Added for clarity in multi-pose output
            "split_strength": round(split_strength, 3),
            "interpretation": "High = Clear Split | Low = Merged/Noisy",
            "ratios": {pos1: round(robust_ratio1, 2), pos2: round(robust_ratio2, 2)},
            "local_densities": {pos1: round(local_size1, 3), pos2: round(local_size2, 3)},
            "cross_dist": round(cross_dist, 3)
        })

    # Return the list of results (or None if no valid pairs were found)
    return results if results else None

In [104]:
for lemma, poses in polycategorial_lexunits.items():
    result = check_robust_split(matrix, idx_to_unit, unit_to_idx, lemma, poses, k_neighbors=20, metric='cosine')
    for dico in result:
        print(dico)

{'lemma': '/', 'poses_compared': ('CCONJ', 'ADP'), 'split_strength': 0.489, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'CCONJ': 0.49, 'ADP': 0.58}, 'local_densities': {'CCONJ': 0.303, 'ADP': 0.255}, 'cross_dist': 0.148}
{'lemma': '1er', 'poses_compared': ('NUM', 'ADJ'), 'split_strength': 6.156, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'NUM': 6.16, 'ADJ': 6.7}, 'local_densities': {'NUM': 0.05, 'ADJ': 0.046}, 'cross_dist': 0.306}
{'lemma': 'allemand', 'poses_compared': ('NOUN', 'ADJ'), 'split_strength': 8.209, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'NOUN': 8.21, 'ADJ': 49.03}, 'local_densities': {'NOUN': 0.066, 'ADJ': 0.011}, 'cross_dist': 0.541}
{'lemma': 'américain', 'poses_compared': ('NOUN', 'ADJ'), 'split_strength': 8.558, 'interpretation': 'High = Clear Split | Low = Merged/Noisy', 'ratios': {'NOUN': 8.56, 'ADJ': 57.61}, 'local_densities': {'NOUN': 0.082, 'ADJ': 0.012}, 'cross_dist': 0.702}

In [103]:
corpus._idx2feature

{0: 'node:X:child:Definite=Def',
 1: 'node:X:child:Definite=Ind',
 2: 'node:X:child:Emph=No',
 3: 'node:X:child:Emph=Yes',
 4: 'node:X:child:ExtPos=ADJ',
 5: 'node:X:child:ExtPos=ADP',
 6: 'node:X:child:ExtPos=ADV',
 7: 'node:X:child:ExtPos=CCONJ',
 8: 'node:X:child:ExtPos=DET',
 9: 'node:X:child:ExtPos=INTJ',
 10: 'node:X:child:ExtPos=NOUN',
 11: 'node:X:child:ExtPos=PRON',
 12: 'node:X:child:ExtPos=PROPN',
 13: 'node:X:child:ExtPos=SCONJ',
 14: 'node:X:child:ExtPos=VERB',
 15: 'node:X:child:Mood=Cnd',
 16: 'node:X:child:Mood=Imp',
 17: 'node:X:child:Mood=Ind',
 18: 'node:X:child:Mood=Sub',
 19: 'node:X:child:NumType=Ord',
 20: 'node:X:child:Number=Plur',
 21: 'node:X:child:Number=Sing',
 22: 'node:X:child:Number__psor=Plur',
 23: 'node:X:child:Number__psor=Sing',
 24: 'node:X:child:Person=1',
 25: 'node:X:child:Person=2',
 26: 'node:X:child:Person=3',
 27: 'node:X:child:Person__psor=1',
 28: 'node:X:child:Person__psor=2',
 29: 'node:X:child:Person__psor=3',
 30: 'node:X:child:Polarit